In [48]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

In [49]:
df = pd.read_csv("../data/Regression_wine_quality_filtered.csv")

In [50]:
y = df["quality"]
X = df.drop(["quality"], axis=1)

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [52]:
lr = LinearRegression().fit(X_train, y_train)

In [53]:
ridge = Ridge().fit(X_train, y_train)

In [54]:
lasso = Lasso().fit(X_train, y_train)

In [55]:
en = ElasticNet().fit(X_train, y_train)

In [56]:
poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('regressor', LinearRegression())
])

poly_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('poly', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06


In [57]:
models_dict = {
    "Linear": lr,
    "Lasso": lasso,
    "Ridge": ridge,
    "ElasticNet": en,
    "Polynomial": poly_model
}

results = []

for name, model in models_dict.items():
    y_pred = model.predict(X_test)
    
    results.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred), 
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

,Model,MSE,MAE,MAPE,R2
0,Linear,0.387848,0.502137,0.089681,0.406512
1,Lasso,0.647319,0.661857,0.118447,0.009467
2,Ridge,0.391100,0.504458,0.090022,0.401536
3,ElasticNet,0.648317,0.659041,0.117971,0.007940
4,Polynomial,0.387695,0.502717,0.090227,0.406746


In [58]:
param_grid_lr = {'fit_intercept': [True, False], 'copy_X': [True, False]}
lr_optimal = GridSearchCV(LinearRegression(), param_grid_lr, cv=5)
lr_optimal.fit(X_train, y_train)
lr_optimal.best_params_

{'copy_X': True, 'fit_intercept': False}

In [59]:
param_grid_ridge = {
    'alpha': np.arange(0, 1, 0.1),
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag']
}

ridge_optimal = RandomizedSearchCV(Ridge(), param_grid_ridge).fit(X_train, y_train)
ridge_optimal.fit(X_train, y_train)
ridge_optimal.best_params_

c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the 

{'solver': 'cholesky', 'alpha': np.float64(0.5)}

In [60]:
def objective_en(trial):
    alpha = trial.suggest_float('alpha', 1e-5, 10.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    en_optimized = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
    return cross_val_score(en_optimized, X_train, y_train, cv=5, scoring='r2').mean()

study_en = optuna.create_study(direction='maximize')
study_en.optimize(objective_en, n_trials=50)

en_optimal = ElasticNet(**study_en.best_params)
en_optimal.fit(X_train, y_train)

[I 2026-03-12 18:12:16,503] A new study created in memory with name: no-name-cc7feb2c-4913-4c0e-bd41-35df50f2d2f3
[I 2026-03-12 18:12:16,515] Trial 0 finished with value: 0.3056940497663627 and parameters: {'alpha': 0.013297870418928064, 'l1_ratio': 0.6248671938771843}. Best is trial 0 with value: 0.3056940497663627.
[I 2026-03-12 18:12:16,526] Trial 1 finished with value: 0.2755172092984398 and parameters: {'alpha': 0.09877207272033246, 'l1_ratio': 0.009646389528956134}. Best is trial 0 with value: 0.3056940497663627.
[I 2026-03-12 18:12:16,537] Trial 2 finished with value: 0.32037725615032964 and parameters: {'alpha': 1.395290767121803e-05, 'l1_ratio': 0.48555320922367706}. Best is trial 2 with value: 0.32037725615032964.
[I 2026-03-12 18:12:16,549] Trial 3 finished with value: 0.3205570248158124 and parameters: {'alpha': 0.00023932423180224853, 'l1_ratio': 0.9325985986270586}. Best is trial 3 with value: 0.3205570248158124.
[I 2026-03-12 18:12:16,559] Trial 4 finished with value: 0.

,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",0.00041970352129358697
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.19658078197489065
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [61]:
def objective_poly(trial):
    degree = trial.suggest_int('degree', 1, 4)
    include_bias = trial.suggest_categorical('include_bias', [True, False])
    model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=include_bias), LinearRegression())
    return cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()

study_poly = optuna.create_study(direction='maximize')
study_poly.optimize(objective_poly, n_trials=10)

bp = study_poly.best_params
poly_optimal = make_pipeline(
    PolynomialFeatures(degree=bp['degree'], include_bias=bp['include_bias']), 
    LinearRegression()
)
poly_optimal.fit(X_train, y_train)

[I 2026-03-12 18:12:17,099] A new study created in memory with name: no-name-7c17060a-a449-48bb-ab2e-c7682d5dd2a5
[I 2026-03-12 18:12:17,117] Trial 0 finished with value: 0.31967483102539407 and parameters: {'degree': 1, 'include_bias': True}. Best is trial 0 with value: 0.31967483102539407.
[I 2026-03-12 18:12:18,240] Trial 1 finished with value: -732.7049144904092 and parameters: {'degree': 4, 'include_bias': False}. Best is trial 0 with value: 0.31967483102539407.
[I 2026-03-12 18:12:19,372] Trial 2 finished with value: -732.7049144904092 and parameters: {'degree': 4, 'include_bias': False}. Best is trial 0 with value: 0.31967483102539407.
[I 2026-03-12 18:12:20,505] Trial 3 finished with value: -732.7049144904092 and parameters: {'degree': 4, 'include_bias': False}. Best is trial 0 with value: 0.31967483102539407.
[I 2026-03-12 18:12:20,547] Trial 4 finished with value: 0.2485251976064072 and parameters: {'degree': 2, 'include_bias': True}. Best is trial 0 with value: 0.31967483102

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('polynomialfeatures', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",1
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06


In [62]:
def objective_lasso(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 1e-6, 10.0, log=True),
        'selection': trial.suggest_categorical('selection', ['cyclic', 'random'])
    }
    model = Lasso(**params, max_iter=10000)
    
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    return score

study_lasso = optuna.create_study(direction='maximize')
study_lasso.optimize(objective_lasso, n_trials=50)

best_lasso_params = study_lasso.best_params
lasso_optimal = Lasso(**best_lasso_params)
lasso_optimal.fit(X_train, y_train)

[I 2026-03-12 18:12:23,442] A new study created in memory with name: no-name-642b380c-8725-4624-b8d0-52a64b1d1238
[I 2026-03-12 18:12:23,453] Trial 0 finished with value: 0.19049523158752535 and parameters: {'alpha': 0.15269660060036822, 'selection': 'random'}. Best is trial 0 with value: 0.19049523158752535.
[I 2026-03-12 18:12:23,465] Trial 1 finished with value: 0.32039289290030676 and parameters: {'alpha': 2.4437986951839407e-05, 'selection': 'random'}. Best is trial 1 with value: 0.32039289290030676.
[I 2026-03-12 18:12:23,475] Trial 2 finished with value: 0.3041151510960421 and parameters: {'alpha': 0.011653064476277588, 'selection': 'cyclic'}. Best is trial 1 with value: 0.32039289290030676.
[I 2026-03-12 18:12:23,486] Trial 3 finished with value: 0.21601685974770257 and parameters: {'alpha': 0.10420840407332768, 'selection': 'cyclic'}. Best is trial 1 with value: 0.32039289290030676.
[I 2026-03-12 18:12:23,497] Trial 4 finished with value: 0.320403865384111 and parameters: {'al

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.000359863740984233
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'random'


In [63]:
models_dict = {
    "Linear": lr_optimal,
    "Lasso": lasso_optimal,
    "Ridge": ridge_optimal,
    "ElasticNet": en_optimal,
    "Polynomial": poly_optimal
}

results = []

for name, model in models_dict.items():
    y_pred = model.predict(X_test)
    
    results.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred), 
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

,Model,MSE,MAE,MAPE,R2
0,Linear,0.388853,0.502505,0.089732,0.404974
1,Lasso,0.389872,0.503358,0.089861,0.403414
2,Ridge,0.389853,0.503369,0.089849,0.403444
3,ElasticNet,0.389981,0.503457,0.089864,0.403249
4,Polynomial,0.387848,0.502137,0.089681,0.406512


In [64]:
def get_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

In [65]:
def get_mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

In [66]:
def get_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [67]:
def get_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

In [68]:
def get_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - (ss_res / ss_tot)

In [69]:
results = []

for name, model in models_dict.items():
    y_pred = model.predict(X_test)
    
    results.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred), 
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
        "Custom MSE": get_mse(y_test, y_pred),
        "Custom MAE": get_mae(y_test, y_pred), 
        "Custom MAPE": get_mape(y_test, y_pred),
        "Custom R2": get_r2(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

,Model,MSE,MAE,MAPE,R2,Custom MSE,Custom MAE,Custom MAPE,Custom R2
0,Linear,0.388853,0.502505,0.089732,0.404974,0.388853,0.502505,0.089732,0.404974
1,Lasso,0.389872,0.503358,0.089861,0.403414,0.389872,0.503358,0.089861,0.403414
2,Ridge,0.389853,0.503369,0.089849,0.403444,0.389853,0.503369,0.089849,0.403444
3,ElasticNet,0.389981,0.503457,0.089864,0.403249,0.389981,0.503457,0.089864,0.403249
4,Polynomial,0.387848,0.502137,0.089681,0.406512,0.387848,0.502137,0.089681,0.406512


In [70]:
from Lab2_LinearRegression import LinearRegressionSGD

model = LinearRegressionSGD(lr=0.001, epochs=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"MSE: {get_mse(y_test, y_pred):.4f}")
print(f"MAPE: {get_mape(y_test, y_pred):.4f}")
print(f"MAE: {get_mae(y_test, y_pred):.4f}")
print(f"R2 Score: {get_r2(y_test, y_pred):.4f}")

MSE: 0.3935
MAPE: 0.0908
MAE: 0.5075
R2 Score: 0.3978


У всех моделей получились +- схожие метрики. Для моделей Lasso и ElasticNet пришлось подбирать гиперпараметры для улучшения метрик.

Формально лучшей моделью для данной задачи является полиномиальная регрессия, так как у нее меньше ошибка и выше коэффициент R^2.